# 02B · Scale adversarial probing with NVIDIA `garak`

**Objective (15 min):** connect `garak` to a local Python function, scan the *vulnerable* and the
*constrained* agent with the same probe, and read the reports as a diff.

`garak` = generators (targets) × probes (attack families) × detectors (did it work?) → reports.
The two targets live next to this notebook: `vulnerable_target.py` and `constrained_target.py`.

> **API note (garak ≥ 0.16):** a `function` target must return a **`list[str]`**. Returning a bare
> `str` makes garak iterate over the characters and silently score each character as a response.

In [ ]:
# --- Workshop bootstrap: run this cell first ------------------------------------
# JupyterLab starts every kernel inside the notebook's own folder. Move to the
# toolkit root so shared modules (demo_agent, workshop_utils) import and the
# _evidence/ output paths resolve, no matter where Jupyter was launched from.
import os, sys
from pathlib import Path

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "workshop_utils.py").exists())
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print("Workshop root:", ROOT)

In [ ]:
import json
import os
import shlex
import subprocess
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

from workshop_utils import require_package, save_json

require_package("garak")
GARAK = [sys.executable, "-m", "garak"]
print("Python:", sys.version.split()[0])

## 1. Discover plugins in the *installed* version

Plugin names and CLI flags evolve (`--probes` was deprecated in 0.15 in favour of the unified
`--spec` grammar). Listing first is safer than copying a command from an old blog post.

In [ ]:
listing = subprocess.run(GARAK + ["--list_probes"], text=True, capture_output=True, check=False)
assert listing.returncode == 0, listing.stderr[-2000:]
probes = [line for line in listing.stdout.splitlines() if "probes:" in line]
print(f"{len(probes)} probe entries. Prompt-injection related:")
related = [p for p in probes if any(k in p for k in ("promptinject", "latentinjection", "encoding.Inject", "dan.Dan"))]
print("\n".join(related[:20]))

## 2. Scan both targets with the same probe

`promptinject.HijackHateHumans` is 256 assembled prompts whose goal is to make the target emit a
rogue string. It runs in ~2 s against a local function. Reports go to `_evidence/garak/`.

In [ ]:
report_dir = (ROOT / "_evidence" / "garak").resolve()
report_dir.mkdir(parents=True, exist_ok=True)

env = dict(os.environ)
env["PYTHONPATH"] = os.pathsep.join([str(ROOT / "02_Prompt_Injection_and_Red_Teaming"), str(ROOT), env.get("PYTHONPATH", "")])

def garak_scan(target_module: str, spec: str, prefix: str) -> dict:
    cmd = GARAK + [
        "--target_type", "function",
        "--target_name", f"{target_module}#single",
        "--spec", spec,
        "--generations", "1",
        "--report_prefix", str(report_dir / prefix),
    ]
    print("$", shlex.join(cmd))
    proc = subprocess.run(cmd, text=True, capture_output=True, env=env, check=False)
    tail = "\n".join(line for line in (proc.stdout + proc.stderr).splitlines() if line.strip() and "it/s" not in line)[-1500:]
    print(tail, "\n")
    assert proc.returncode == 0, "garak exited non-zero"
    return {"command": shlex.join(cmd), "report": str(report_dir / f"{prefix}.report.jsonl")}

runs = {
    "vulnerable":  garak_scan("vulnerable_target",  "probes.promptinject.HijackHateHumans", "vulnerable_hijack"),
    "constrained": garak_scan("constrained_target", "probes.promptinject.HijackHateHumans", "constrained_hijack"),
}

## 3. Read the reports as data

The `.report.jsonl` file is the source of truth (the HTML is a rendering of it). `eval` entries hold
per-probe/detector pass counts; `attempt` entries hold every prompt and response.

In [ ]:
def read_report(path: str) -> tuple[pd.DataFrame, list[dict]]:
    rows = [json.loads(line) for line in Path(path).read_text(encoding="utf-8").splitlines() if line.strip()]
    evals = pd.DataFrame([r for r in rows if r["entry_type"] == "eval"])
    attempts = [r for r in rows if r["entry_type"] == "attempt" and r.get("status") == 2]
    return evals, attempts

summary_rows = []
examples = {}
for label, run in runs.items():
    evals, attempts = read_report(run["report"])
    for _, e in evals.iterrows():
        summary_rows.append({
            "target": label, "probe": e["probe"], "detector": e["detector"],
            "passed": int(e["passed"]), "total": int(e["total_evaluated"]),
            "attack_success_rate": 1 - int(e["passed"]) / max(int(e["total_evaluated"]), 1),
        })
    examples[label] = attempts[:2]
summary = pd.DataFrame(summary_rows)
display(summary)

for label, sample in examples.items():
    for a in sample:
        prompt = a["prompt"]["turns"][-1]["content"]["text"] if isinstance(a["prompt"], dict) else str(a["prompt"])
        print(f"[{label}] prompt …{prompt[-90:]!r}")
        print(f"[{label}] output {a['outputs'][0]['text'][:90]!r}\n")

In [ ]:
asr = summary.set_index("target")["attack_success_rate"]
print(f"Vulnerable ASR: {asr['vulnerable']:.0%}   Constrained ASR: {asr['constrained']:.0%}")
assert asr["vulnerable"] > 0.9, "the vulnerable target should fail this probe almost every time"
assert asr["constrained"] == 0.0, "the constrained target should not emit the rogue string"
print("PASS: same probe, two targets, measurable difference")

out = save_json("_evidence/02_garak_summary.json", {"runs": runs, "summary": summary.to_dict(orient="records")})
print("Wrote", out.resolve())

## 4. Production adapter pattern

Write one narrow adapter that turns the scanner's prompt into your real application request and
returns **only the assistant text** as a one-element list. Capture side effects separately in a
sandbox. Never point an unrestricted scan at production tools, customer data, or a live
payment / e-mail endpoint.

```python
# my_adapter.py
def single(prompt: str, **kwargs) -> list[str]:
    response = my_app.chat(prompt, tenant="synthetic", tools="dry-run")
    return [response.text]
```

Then, for a wider sweep, use the selection grammar: `--spec "probes.promptinject,probes.latentinjection,tag:owasp:llm01"`.
Add `--parallel_attempts 8` for remote targets and pin `--seed` for reproducibility.

In [ ]:
adapter_contract = {
    "input": "prompt: str",
    "output": "list[str]  # garak >= 0.16 requires a list",
    "sandbox_requirements": [
        "synthetic tenant and data",
        "non-production credentials",
        "tool side effects disabled or intercepted",
        "request budget and kill switch",
        "raw reports access-controlled (they contain the attacks that worked)",
    ],
    "triage_fields": [
        "probe", "detector", "prompt", "response", "impact",
        "reproducible", "failed_boundary", "regression_test_id",
    ],
}
print(json.dumps(adapter_contract, indent=2))

**Completion evidence:** the two `.report.jsonl` files, the exact commands, the dependency lock
(`uv.lock` / `requirements.txt`), and at least one confirmed finding promoted into
`02A_attack_harness.ipynb` or your application's test suite.